
# Análise dos resultados do TCC — Entanglion

Este notebook foi estruturado para gerar:

1. **Tabela de participação por aula**
2. **Tabela de ganho conceitual (pré × pós)**
3. **Gráfico de médias pré e pós**
4. **Tabela Likert média (clareza, ritmo, interação)**

## Estrutura esperada do repositório

Coloque este notebook na raiz do projeto ou em uma pasta como `analise/`, e deixe o arquivo Excel acessível no caminho configurado na célula de parâmetros.

Exemplo:

```text
tcc-gbl-entanglion-computacao-quantica/
├── data/
│   └── Resultados.xlsx
├── notebooks/
│   └── analise_resultados_tcc.ipynb
└── README.md
```

## Observação metodológica

As questões de **pré** e **pós** no arquivo são abertas (`O que você sabe...` / `O que você aprendeu...`).  
Por isso, o cálculo de **ganho conceitual** precisa de uma **codificação/rúbrica**.

Este notebook já deixa preparado o fluxo para:

- extrair as respostas;
- gerar uma base de codificação;
- calcular médias, delta, teste estatístico e tamanho de efeito depois que os escores forem atribuídos.


In [ ]:

# Instale pacotes se necessário:
# !pip install pandas openpyxl scipy matplotlib notebook

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.stats import wilcoxon, ttest_rel


## 1. Parâmetros

In [ ]:

# Ajuste este caminho conforme a estrutura do seu repositório
ARQUIVO_XLSX = Path("../data/Resultados.xlsx")  # exemplo quando o notebook está em /notebooks
# ARQUIVO_XLSX = Path("data/Resultados.xlsx")   # exemplo quando o notebook está na raiz

AULAS = [f"Aula {i}" for i in range(1, 7)]

# Metadados manuais para a tabela de participação
meta_participacao = pd.DataFrame([
    {"Aula": 1, "Data": "14/04/2025", "Local": "Embarque Digital", "Previsto": 30, "Presentes": 8,  "Respondentes": 8},
    {"Aula": 2, "Data": "26/04/2025", "Local": "CESAR School",     "Previsto": 19, "Presentes": 6,  "Respondentes": 6},
    {"Aula": 3, "Data": "29/04/2025", "Local": "ETE (Turma C)",    "Previsto": 45, "Presentes": 40, "Respondentes": 5},
    {"Aula": 4, "Data": "30/04/2025", "Local": "ETE (Turma B)",    "Previsto": 43, "Presentes": 20, "Respondentes": 4},
    {"Aula": 5, "Data": "10/05/2025", "Local": "CESAR School",     "Previsto": 6,  "Presentes": 4,  "Respondentes": 4},
    {"Aula": 6, "Data": "24/05/2025", "Local": "CESAR School",     "Previsto": 16, "Presentes": 8,  "Respondentes": 8},
])

meta_participacao["Taxa (%)"] = (meta_participacao["Respondentes"] / meta_participacao["Previsto"] * 100).round(1)
meta_participacao


## 2. Leitura do arquivo Excel

In [ ]:

if not ARQUIVO_XLSX.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {ARQUIVO_XLSX}\n"
        "Ajuste o caminho em `ARQUIVO_XLSX`."
    )

xls = pd.ExcelFile(ARQUIVO_XLSX)
xls.sheet_names


In [ ]:

dados_aulas = {aba: xls.parse(aba) for aba in AULAS if aba in xls.sheet_names}

for nome, df in dados_aulas.items():
    print(f"\n===== {nome} =====")
    print(df.head(2))
    print(f"Dimensão: {df.shape}")


## 3. Tabela de participação

In [ ]:

tabela_participacao = meta_participacao.copy()
tabela_participacao


In [ ]:

def formatar_latex_participacao(df):
    linhas = []
    linhas.append(r"\begin{table}[ht]")
    linhas.append(r"\centering")
    linhas.append(r"\caption{Participação por aula.}")
    linhas.append(r"\label{tab:participacao}")
    linhas.append(r"\begin{tabular}{l l l r r r r}")
    linhas.append(r"\toprule")
    linhas.append(r"Aula & Data & Local & Previsto & Presentes & Respondentes & Taxa\;(\%) \\")
    linhas.append(r"\midrule")
    for _, row in df.iterrows():
        taxa = str(row["Taxa (%)"]).replace(".", ",")
        linhas.append(
            f'{int(row["Aula"])} & {row["Data"]} & {row["Local"]} & '
            f'{int(row["Previsto"])} & {int(row["Presentes"])} & {int(row["Respondentes"])} & {taxa} \\\\'
        )
    linhas.append(r"\bottomrule")
    linhas.append(r"\end{tabular}")
    linhas.append(r"\end{table}")
    return "\n".join(linhas)

latex_participacao = formatar_latex_participacao(tabela_participacao)
print(latex_participacao)


## 4. Avaliação Likert (clareza, ritmo, interação)

In [ ]:

COL_CLAREZA = "Como você avalia a clareza da explicação dos conteúdos abordados na aula?"
COL_RITMO = "O ritmo da aula foi adequado para o seu entendimento?"
COL_INTERACAO = "Como você avalia a interação e a dinâmica da aula?"

map_clareza = {
    "Muito clara": 5,
    "Clara": 4,
    "Pouco clara": 2,
    "Nada clara": 1
}

map_ritmo = {
    "Muito adequado": 5,
    "Adequado": 4,
    "Pouco adequado": 2,
    "Inadequado": 1,
    "Muito lento": 2,
    "Muito rápido": 2
}

map_interacao = {
    "Muito interativa e envolvente": 5,
    "Interativa na medida certa": 4,
    "Pouca interação": 2,
    "Nenhuma interação": 1
}


In [ ]:

likert_rows = []

for nome, df in dados_aulas.items():
    aula_num = int(nome.split()[-1])

    serie_clareza = df[COL_CLAREZA].map(map_clareza) if COL_CLAREZA in df.columns else pd.Series(dtype=float)
    serie_ritmo = df[COL_RITMO].map(map_ritmo) if COL_RITMO in df.columns else pd.Series(dtype=float)
    serie_interacao = df[COL_INTERACAO].map(map_interacao) if COL_INTERACAO in df.columns else pd.Series(dtype=float)

    likert_rows.append({
        "Aula": aula_num,
        "Clareza": round(serie_clareza.dropna().mean(), 2) if not serie_clareza.dropna().empty else np.nan,
        "Ritmo": round(serie_ritmo.dropna().mean(), 2) if not serie_ritmo.dropna().empty else np.nan,
        "Interação": round(serie_interacao.dropna().mean(), 2) if not serie_interacao.dropna().empty else np.nan,
    })

tabela_likert = pd.DataFrame(likert_rows).sort_values("Aula")
tabela_likert


In [ ]:

def formatar_latex_likert(df):
    linhas = []
    linhas.append(r"\begin{table}[ht]")
    linhas.append(r"\centering")
    linhas.append(r"\caption{Avaliação Likert média (1--5) por aula.}")
    linhas.append(r"\label{tab:likert}")
    linhas.append(r"\begin{tabular}{lrrr}")
    linhas.append(r"\toprule")
    linhas.append(r"Aula & Clareza & Ritmo & Interação \\")
    linhas.append(r"\midrule")
    for _, row in df.iterrows():
        clareza = f'{row["Clareza"]:.2f}'.replace(".", ",") if pd.notna(row["Clareza"]) else "-"
        ritmo = f'{row["Ritmo"]:.2f}'.replace(".", ",") if pd.notna(row["Ritmo"]) else "-"
        interacao = f'{row["Interação"]:.2f}'.replace(".", ",") if pd.notna(row["Interação"]) else "-"
        linhas.append(f'{int(row["Aula"])} & {clareza} & {ritmo} & {interacao} \\\\')
    linhas.append(r"\bottomrule")
    linhas.append(r"\end{tabular}")
    linhas.append(r"\end{table}")
    return "\n".join(linhas)

latex_likert = formatar_latex_likert(tabela_likert)
print(latex_likert)



## 5. Ganho conceitual (pré × pós)

### Importante
Como as respostas são abertas, você precisa **atribuir escores** a cada resposta.

Uma forma prática é usar uma rúbrica simples, por exemplo:

- **0** = em branco / não sabe / irrelevante
- **1** = menção vaga ou muito superficial
- **2** = definição parcialmente correta
- **3** = definição correta e minimamente elaborada
- **4** = definição correta + relação conceitual mais robusta

Abaixo o notebook extrai as respostas e gera uma planilha-base para codificação manual.


In [ ]:

prefixos_pre = [
    "O que você sabe sobre computação?",
    "O que você sabe sobre a palavra quântica?",
    "O que você sabe sobre Computação Quântica?",
    "O que você sabe sobre qubits?",
    "O que você sabe sobre superposição?",
    "O que você sabe sobre entrelaçamento?",
    "O que você sabe sobre portas lógicas quânticas?",
    "O que você sabe sobre circuitos quânticos?",
]

prefixos_pos = [
    "O que você aprendeu sobre a palavra quântica?",
    "O que você aprendeu sobre Computação Quântica?",
    "O que você aprendeu sobre qubits?",
    "O que você aprendeu sobre superposição?",
    "O que você aprendeu sobre entrelaçamento?",
    "O que você aprendeu sobre portas lógicas quânticas?",
    "O que você aprendeu sobre circuitos quânticos?",
]


In [ ]:

# Gera uma base longa para codificação manual
linhas_codificacao = []

for nome, df in dados_aulas.items():
    aula_num = int(nome.split()[-1])

    for idx, row in df.iterrows():
        participante = f"A{aula_num}_P{idx+1:03d}"

        for pergunta in prefixos_pre:
            if pergunta in df.columns:
                linhas_codificacao.append({
                    "Aula": aula_num,
                    "Participante": participante,
                    "Momento": "Pré",
                    "Pergunta": pergunta,
                    "Resposta": row.get(pergunta, np.nan),
                    "Escore": np.nan,
                })

        for pergunta in prefixos_pos:
            if pergunta in df.columns:
                linhas_codificacao.append({
                    "Aula": aula_num,
                    "Participante": participante,
                    "Momento": "Pós",
                    "Pergunta": pergunta,
                    "Resposta": row.get(pergunta, np.nan),
                    "Escore": np.nan,
                })

base_codificacao = pd.DataFrame(linhas_codificacao)
base_codificacao.head(20)


In [ ]:

# Salve esta base, preencha a coluna 'Escore' manualmente e depois reimporte.
# Exemplo:
# base_codificacao.to_excel("../data/base_codificacao_conceitual.xlsx", index=False)

base_codificacao



### 5.1 Reimportar a base já codificada

Depois de preencher a coluna `Escore`, execute a célula abaixo.


In [ ]:

ARQUIVO_CODIFICADO = Path("../data/base_codificacao_conceitual_preenchida.xlsx")

# Descomente quando o arquivo já existir:
# codificado = pd.read_excel(ARQUIVO_CODIFICADO)
# codificado.head()


In [ ]:

def cohen_d_pareado(pre, pos):
    diff = pos - pre
    sd_diff = diff.std(ddof=1)
    if sd_diff == 0 or np.isnan(sd_diff):
        return np.nan
    return diff.mean() / sd_diff

def resumo_ganho_conceitual(codificado):
    # Soma de escores por participante em cada momento
    soma = (
        codificado
        .dropna(subset=["Escore"])
        .groupby(["Aula", "Participante", "Momento"], as_index=False)["Escore"]
        .sum()
    )

    pre = soma[soma["Momento"] == "Pré"].rename(columns={"Escore": "Pre"})
    pos = soma[soma["Momento"] == "Pós"].rename(columns={"Escore": "Pos"})

    pareado = pre.merge(pos, on=["Aula", "Participante"], how="inner")

    resultados = []

    for aula, grupo in pareado.groupby("Aula"):
        pre_vals = grupo["Pre"].astype(float)
        pos_vals = grupo["Pos"].astype(float)
        n = len(grupo)
        delta = pos_vals.mean() - pre_vals.mean()

        # Regra simples:
        # n pequeno -> Wilcoxon
        # n maior -> t pareado
        if n < 10:
            try:
                stat, p = wilcoxon(pre_vals, pos_vals)
                teste = "Wilcoxon"
            except ValueError:
                stat, p = np.nan, np.nan
                teste = "Wilcoxon"
        else:
            stat, p = ttest_rel(pre_vals, pos_vals)
            teste = "t pareado"

        efeito = cohen_d_pareado(pre_vals, pos_vals)

        resultados.append({
            "Aula": aula,
            "n": n,
            "Média pré": round(pre_vals.mean(), 2),
            "Média pós": round(pos_vals.mean(), 2),
            "Δ": round(delta, 2),
            "Teste": teste,
            "p": round(float(p), 4) if pd.notna(p) else np.nan,
            "Efeito": round(float(efeito), 2) if pd.notna(efeito) else np.nan,
        })

    return pd.DataFrame(resultados).sort_values("Aula"), pareado


In [ ]:

# Exemplo de uso:
# tabela_ganho, base_pareada = resumo_ganho_conceitual(codificado)
# tabela_ganho

def formatar_latex_ganho(df):
    linhas = []
    linhas.append(r"\begin{table}[ht]")
    linhas.append(r"\centering")
    linhas.append(r"\caption{Ganho conceitual (pré $\times$ pós) por aula.}")
    linhas.append(r"\label{tab:ganho_conceitual}")
    linhas.append(r"\begin{tabular}{lrrrrrcr}")
    linhas.append(r"\toprule")
    linhas.append(r"Aula & n & Média pré & Média pós & $\Delta$ & Teste & $p$ & Efeito \\")
    linhas.append(r"\midrule")
    for _, row in df.iterrows():
        media_pre = f'{row["Média pré"]:.2f}'.replace(".", ",")
        media_pos = f'{row["Média pós"]:.2f}'.replace(".", ",")
        delta = f'{row["Δ"]:.2f}'.replace(".", ",")
        p = f'{row["p"]:.4f}'.replace(".", ",") if pd.notna(row["p"]) else "-"
        efeito = f'{row["Efeito"]:.2f}'.replace(".", ",") if pd.notna(row["Efeito"]) else "-"
        linhas.append(
            f'{int(row["Aula"])} & {int(row["n"])} & {media_pre} & {media_pos} & '
            f'{delta} & {row["Teste"]} & {p} & {efeito} \\\\'
        )
    linhas.append(r"\bottomrule")
    linhas.append(r"\end{tabular}")
    linhas.append(r"\end{table}")
    return "\n".join(linhas)

# Exemplo:
# print(formatar_latex_ganho(tabela_ganho))


## 6. Gráfico pré × pós

In [ ]:

def plot_ganho_pre_pos(tabela_ganho):
    x = np.arange(len(tabela_ganho))
    largura = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - largura/2, tabela_ganho["Média pré"], largura, label="Pré")
    ax.bar(x + largura/2, tabela_ganho["Média pós"], largura, label="Pós")

    ax.set_xticks(x)
    ax.set_xticklabels(tabela_ganho["Aula"].astype(str))
    ax.set_xlabel("Aula")
    ax.set_ylabel("Média de escores")
    ax.set_title("Ganho conceitual médio antes e depois da intervenção")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()

# Exemplo:
# plot_ganho_pre_pos(tabela_ganho)



## 7. Texto LaTeX da figura

Use algo assim no seu texto:

```latex
\begin{figure}[ht!]
\centering
\includegraphics[width=0.85\linewidth]{figuras/ganho_prepos.png}
\caption{Ganho conceitual médio antes e depois da intervenção.}
\label{fig:ganho_prepos}
\end{figure}
```



## 8. Próximos passos sugeridos

1. Ajustar o caminho do arquivo `Resultados.xlsx`;
2. Executar a extração da `base_codificacao`;
3. Preencher os **escores** da rúbrica;
4. Reimportar a base preenchida;
5. Gerar:
   - tabela de ganho conceitual;
   - gráfico pré/pós;
   - LaTeX final para o capítulo de resultados.
